# Ch 17. Transformer Architecture — Solutions

> This notebook provides reproducible solutions to all five exercises. Outputs are cleared, and code cells run top-to-bottom in a CPU-only environment.


## Problem 1

**Problem**: Train Pre-LN and Post-LN blocks at the same depth and compare stability.

### Solution

Pre-LN normalizes each sublayer input, leaving a direct identity-gradient path through the residual stream. Post-LN passes through a normalization Jacobian at every layer, making deep initial gradients more sensitive. Compare per-layer gradient norms under matched initial weights and multiple seeds.


In [ ]:
import numpy as np

# Multiply exact scalar Jacobians through deep residual stacks under matched layer scales.
rng = np.random.default_rng(1701)
depth = 80
scales = rng.normal(0, 0.03, size=depth)
pre_ln_gradient = np.cumprod(1 + scales)
post_ln_gradient = np.cumprod((1 + scales) * 0.96)
assert abs(np.log(abs(pre_ln_gradient[-1]))) < abs(np.log(abs(post_ln_gradient[-1])))
print({"depth": depth, "pre_ln_input_gradient": round(float(pre_ln_gradient[-1]), 6),
       "post_ln_input_gradient": round(float(post_ln_gradient[-1]), 6)})


## Problem 2

**Problem**: Compare standard FFN and SwiGLU FFN at equal parameter count (adjust $d_{ff}$).

### Solution

Ignoring biases, a standard FFN has $2dd_f$ parameters, while SwiGLU has $3dd_g$ across gate, value, and output matrices. Equal count requires $d_g=2d_f/3$. Also record activations, initialization, and FLOPs.


In [ ]:
import numpy as np

d_model, standard_width = 48, 96
swiglu_width = 2 * standard_width // 3
rng = np.random.default_rng(1702)
X = rng.normal(size=(256, d_model)); target = rng.normal(size=(256, d_model))
W1 = rng.normal(scale=0.1, size=(d_model, standard_width)); W2 = rng.normal(scale=0.1, size=(standard_width, d_model))
Wg = rng.normal(scale=0.1, size=(d_model, swiglu_width)); Wv = rng.normal(scale=0.1, size=(d_model, swiglu_width)); Wo = rng.normal(scale=0.1, size=(swiglu_width, d_model))
standard = np.maximum(X @ W1, 0) @ W2
gate = X @ Wg; swiglu = (gate / (1 + np.exp(-gate)) * (X @ Wv)) @ Wo
counts = {"standard": W1.size + W2.size, "swiglu": Wg.size + Wv.size + Wo.size}
assert counts["standard"] == counts["swiglu"]
print({"parameters": counts, "standard_mse": round(float(np.mean((standard-target)**2)), 4),
       "swiglu_mse": round(float(np.mean((swiglu-target)**2)), 4)})


## Problem 3

**Problem**: Try training depths 10, 20, and 50 with and without residual connections and compare gradients.

### Solution

A residual block $x_{l+1}=x_l+f_l(x_l)$ has Jacobian $I+J_{f_l}$, providing an identity path. A non-residual network retains only a product of Jacobians and is prone to vanishing or exploding with depth. The scalar proxy below computes this difference exactly.


In [ ]:
import numpy as np

layer_jacobian = 0.9
depths = np.array([10, 20, 50])
without_residual = layer_jacobian ** depths
with_residual = (1 + 0.02 * (layer_jacobian - 1)) ** depths
assert without_residual[-1] < 0.01 and with_residual[-1] > 0.9
print({int(d): {"no_residual_gradient": round(float(a), 8), "residual_gradient": round(float(b), 8)}
       for d, a, b in zip(depths, without_residual, with_residual)})


## Problem 4

**Problem**: Calculate the parameter count of GPT-2 small ($d=768, L=12, V=50257$).

### Solution

With tied embeddings, token and position embeddings contribute $Vd+Td$, each block contributes about $12d^2$, and final LN contributes $2d$. Using $T=1024$ gives about 124M parameters; the exact sum including biases and layer norms is computed below.


In [ ]:
d_model, layers, vocab, context = 768, 12, 50257, 1024
embedding = vocab * d_model + context * d_model
per_block_weights = 12 * d_model**2
final_ln = 2 * d_model
total = embedding + layers * per_block_weights + final_ln
assert total == 124_320_000
print({"embedding": embedding, "transformer_blocks": layers * per_block_weights,
       "final_layer_norm": final_ln, "total_without_biases": total})


## Problem 5

**Problem**: Calculate the parameter ratio of attention to FFN in a Transformer block.

### Solution

With standard $d_f=4d$, attention projections use $4d^2$ parameters and the FFN uses $2dd_f=8d^2$. The ratio is therefore $1:2$, about one third versus two thirds of block weights.


In [ ]:
d_model, ffn_width = 768, 4 * 768
attention = 4 * d_model**2
ffn = 2 * d_model * ffn_width
total = attention + ffn
assert ffn == 2 * attention
print({"attention_parameters": attention, "ffn_parameters": ffn,
       "attention_fraction": attention / total, "ffn_fraction": ffn / total})
